# AutoLoader (Bronze ingestion)

Incrementally ingests the raw JSON files landed by the **Landing** notebook into the bronze Delta table.

* **Source:** `/Volumes/football_col_dev/landing/raw/api_football/fixtures/` (JSON files)
* **Target:** `football_col_dev.bronze.api_fixtures` (streaming append + metadata columns)

**How it works**
1. Loads shared setup (imports + logger) via `%run ../common/01_Setup`.
2. Auto Loader (`cloudFiles`) picks up new files; schema evolution is tracked in `_schemas/fixtures`.
3. Appends rows plus `_source_file` and `_ingested_at` metadata to bronze, checkpointed at `_checkpoints/bronze_fixtures`.
4. Trigger `availableNow=True` processes all pending files and then stops. Rows written are logged.

**Run:** top to bottom, after the Landing notebook has landed new files.


In [0]:
%run ../common/02_Config

In [0]:
%run ../common/01_Setup

In [0]:
logger = get_logger("autoloader")

# Configuration comes from ../common/02_Config: FIXTURES_DIR, SCHEMA_LOCATION, BRONZE_TABLE.
target_table = BRONZE_TABLE

logger.info(f"Bronze ingestion run started | source: {FIXTURES_DIR}/ | target: {target_table}")

df_raw = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
        .load(f"{FIXTURES_DIR}/")
)



In [0]:
bronze_query = (
    df_raw.withColumns(
        {
            "_source_file": F.col("_metadata.file_path"),
            "_file_modified_at": F.col("_metadata.file_modification_time"),
            "_ingested_at": F.current_timestamp(),
        }
    )
    .writeStream
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .toTable(target_table)
)

bronze_query.awaitTermination()

rows_written = (
    bronze_query.lastProgress["numInputRows"]
    if bronze_query.lastProgress else 0
)
logger.info(f"Bronze ingestion completed | {rows_written} new rows written to {target_table}")


In [0]:
# Preview the bronze table (table name comes from 02_Config)
display(spark.table(BRONZE_TABLE))